# NEX-GDDP Comparison
In this notebook we compare our BCSD downscaled dataset against NASA's NEX-GDDP dataset. While our training datasets differ (NEX-GDDP used the Princeton Global Forcing Dataset and we used ERA5), the comparison will still help contextualize how the datasets stack up and support confidence-building in the dataset's implementation.

This notebook looks at:
- maps of different metrics
- timeseries for a set of individual locations to see pixel-level timeseries
- CDFs of individual locations to assess pixel-level performance


In [1]:
%load_ext autoreload
%autoreload 2
import icechunk
import matplotlib.pyplot as plt
import xarray as xr

from srm import catalog
from srm.plotting import (
    calculate_statistic_to_plot,
    locations,
    plot_4regions_comparisons,
    plot_comparisons,
    plot_timeseries,
    prep_datasets_for_seasonal_cycle_plotting,
    random_non_leap_year,
)

In [ ]:
def icechunk_storage(path: str):
    """Create icechunk Storage from an S3 or local path."""
    if path.startswith("s3://"):
        path_no_scheme = path[len("s3://") :]
        bucket, _, prefix = path_no_scheme.partition("/")
        return icechunk.s3_storage(bucket=bucket, prefix=prefix)
    else:
        return icechunk.local_filesystem_storage(path=path)

In [ ]:
def open_from_icechunk(path: str) -> xr.Dataset:
    """Open a dataset from an icechunk store."""
    storage = icechunk_storage(path)
    icechunk.Repository.open(storage)
    repo = icechunk.Repository.open(storage)
    session = repo.readonly_session("main")
    return xr.open_dataset(session.store, engine="zarr", consolidated=False)

In [ ]:
# historical
ds1 = open_from_icechunk(
    "s3://carbonplan-scratch/srm/outputs/qa/pr-186-39db5bb-nonparametric_hybrid/historical/CESM2-WACCM/tas/r1i1p1f1/global/5e6b6f6c/historical.icechunk"
)
# obs_regridded = open_from_icechunk("s3://carbonplan-scratch/srm/bcsd-cache/qa/pr-186-39db5bb-nonparametric_hybrid/obs/CESM2-WACCM/tas/global/obs_regridded.icechunk/")
# nasa-nex
ds2 = catalog.get("NASA-NEX-historical").to_xarray()
